# Silver IQSS Data Exploration

This notebook explores the **cleaned IQSS data** from the Silver layer.

This data has been processed by `transform/iqss_cleaner.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append('..')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

## 1. Select Year and Load Data

In [ ]:
# Configuration
YEAR = 2023  # Change this to explore different years

# Load silver/cleaned data
filepath = Path(f'../data/silver/iqss/{YEAR}/resultats_iqss_{YEAR}.csv')

if not filepath.exists():
    print(f"❌ No cleaned IQSS data found for year {YEAR}")
    print(f"   Run: python ../main.py --years {YEAR}")
else:
    df_silver = pd.read_csv(filepath, sep=';')
    print(f"📁 Loaded: {filepath.name}")
    print(f"📊 Shape: {df_silver.shape}")

In [ ]:
# Column overview
print(f"📋 Columns ({len(df_silver.columns)}):")
for i, col in enumerate(df_silver.columns, 1):
    print(f"  {i:2}. {col}")

## 2. Data Quality Overview

In [ ]:
# Basic info
print("📈 Dataset Statistics:")
print(f"  Total records: {len(df_silver):,}")
print(f"  Total columns: {len(df_silver.columns)}")
print(f"  Memory usage: {df_silver.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Data types
print("\n📝 Data Types:")
print(df_silver.dtypes.value_counts())

In [ ]:
# Display first few rows
print("📋 First 5 Rows:")
df_silver.head()

## 3. Transformation Verification

In [ ]:
# Check column name normalization (should be lowercase with underscores)
print("✅ Column Naming Convention:")
print(f"  All lowercase? {all(c == c.lower() for c in df_silver.columns)}")
print(f"  Using underscores? {any('_' in c for c in df_silver.columns)}")
print(f"  Any spaces? {any(' ' in c for c in df_silver.columns)}")

In [ ]:
# Check that 'evolution' column was removed
print(f"\n✅ Evolution column removed? {'evolution' not in df_silver.columns}")

In [ ]:
# Check categorical cleaning (Obligatoire/Facultatif, Oui/Non)
categorical_cols = ['participation', 'depot']

for col in categorical_cols:
    if col in df_silver.columns:
        print(f"\n📊 {col.upper()} Values (After Cleaning):")
        print(df_silver[col].value_counts())

        # Check for numbered prefixes (should be removed)
        has_prefixes = df_silver[col].str.contains('1-|2-', na=False).any()
        print(f"  ✅ Number prefixes removed? {not has_prefixes}")

In [ ]:
# Check FINESS normalization (should be 9 digits)
if 'finess' in df_silver.columns:
    print("\n🔢 FINESS Identifier:")
    finess_lengths = df_silver['finess'].astype(str).str.len()
    print(f"  Min length: {finess_lengths.min()}")
    print(f"  Max length: {finess_lengths.max()}")
    print(f"  All 9 digits? {(finess_lengths == 9).all()}")
    print(f"  Sample: {df_silver['finess'].head(5).tolist()}")

## 4. Numeric Column Analysis

In [ ]:
# Identify numeric columns
nb_cols = [c for c in df_silver.columns if c.startswith('nb_')]
score_cols = [c for c in df_silver.columns if c.startswith('score_')]

print("📊 Numeric Columns:")
print(f"  Count columns (nb_*): {len(nb_cols)}")
print(f"  Score columns (score_*): {len(score_cols)}")

# Verify they are numeric types
if nb_cols:
    print(f"\n  nb_* columns are numeric? {all(pd.api.types.is_numeric_dtype(df_silver[c]) for c in nb_cols)}")
if score_cols:
    print(f"  score_* columns are numeric? {all(pd.api.types.is_numeric_dtype(df_silver[c]) for c in score_cols)}")

In [ ]:
# Sample numeric statistics
if score_cols:
    print("\n📊 Sample Score Statistics (first 3 columns):")
    for col in score_cols[:3]:
        print(f"\n{col}:")
        print(df_silver[col].describe())

## 5. Data Completeness

In [ ]:
# Null value analysis
null_counts = df_silver.isnull().sum()
null_pct = (null_counts / len(df_silver) * 100).round(2)

completeness = pd.DataFrame({
    'Column': df_silver.columns,
    'Non_Null': (len(df_silver) - null_counts).values,
    'Null': null_counts.values,
    'Null_%': null_pct.values
}).sort_values('Null_%', ascending=False)

print("🔍 Data Completeness (Top 10 by nulls):")
print(completeness.head(10))

## 6. Sample Records

In [ ]:
# Key columns for display
key_cols = ['finess', 'rs'] if 'rs' in df_silver.columns else ['finess']
if 'participation' in df_silver.columns:
    key_cols.append('participation')
if 'depot' in df_silver.columns:
    key_cols.append('depot')

# Add a sample score column
if score_cols:
    key_cols.append(score_cols[0])

print("🏥 Sample Records (Random 5):")
df_silver[key_cols].sample(min(5, len(df_silver)))

## 7. Quality Score Distribution

In [ ]:
# Distribution of quality scores
if score_cols:
    print("📊 Quality Score Ranges (first 3 score columns):")
    for col in score_cols[:3]:
        valid_scores = df_silver[col].dropna()
        if len(valid_scores) > 0:
            print(f"\n{col}:")
            print(f"  Min: {valid_scores.min():.2f}")
            print(f"  Max: {valid_scores.max():.2f}")
            print(f"  Mean: {valid_scores.mean():.2f}")
            print(f"  Median: {valid_scores.median():.2f}")